### TASK 3: RECOMMENDATION SYSTEM ANALYSIS

In [2]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
data_dir = project_root / 'data'
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Load cleaned dataset
df = pd.read_csv(data_dir / 'cleaned_netflix_data.csv')

# Make sure required columns exist before using them
for col in ['title', 'type', 'director', 'country', 'listed_in']:
    if col not in df.columns:
        df[col] = ''

if 'cast' not in df.columns:
    df['cast'] = ''

if 'description' not in df.columns:
    df['description'] = ''

# Step 1: Extract relevant content features
df['combined_features'] = (
    df['title'].fillna('').astype(str) + ' ' +
    df['type'].fillna('').astype(str) + ' ' +
    df['director'].fillna('').astype(str) + ' ' +
    df['cast'].fillna('').astype(str) + ' ' +
    df['country'].fillna('').astype(str) + ' ' +
    df['listed_in'].fillna('').astype(str) + ' ' +
    df['description'].fillna('').astype(str)
)

# Step 2: Perform text preprocessing & TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['combined_features'])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

# Step 3: Calculate content similarity matrix
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Index mapping for fast lookup
indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

# Step 4: Generate recommendations for selected titles
def recommend_content(title, top_n=5):
    title_clean = title.strip().lower()
    if title_clean not in indices:
        return f"Title '{title}' not found in the dataset."
    
    idx = indices[title_clean]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
        
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    results = df.iloc[movie_indices][['title', 'type', 'listed_in', 'description']].copy()
    results['similarity_score'] = [round(i[1], 4) for i in sim_scores]
    return results

# Step 5: Evaluate recommendation quality
sample_title = df['title'].iloc[0]
print(f"\nTarget Title: '{sample_title}'")
print(f"Category: {df['listed_in'].iloc[0]}\n")
print("Top Recommendations:")
recommendations = recommend_content(sample_title, top_n=5)
print(recommendations[['title', 'type', 'similarity_score']])

TF-IDF Matrix Shape: (8790, 5000)

Target Title: 'Dick Johnson Is Dead'
Category: Documentaries

Top Recommendations:
           title   type  similarity_score
4738    Daffedar  Movie            0.6384
5795    S.W.A.T.  Movie            0.4755
2670  Avengement  Movie            0.4612
2026   Honeytrap  Movie            0.4501
2964     Juanita  Movie            0.4483
